# 🧬 Pharmacy Intelligence Platform — Capstone Clinical Demo

**Author**: 3rd-Year Pharmacy Student | Computational Pharmacogenomics  
**Course Track**: PHAI-101 through PHAI-105 & Capstone  
**Focus**: Drug-Drug Interactions (DDI), Cockcroft-Gault Renal Dosing, & CPIC Pharmacogenomic Risk Stratification  

---

### 🎯 Objectives
1. **Evaluate DDI & PGx Phenotype Interactions**: Detect contraindications when combining substrates and inhibitors (e.g., Codeine + Fluoxetine in CYP2D6 Poor Metabolizers).
2. **Calculate Precision Dosing**: Apply weight-based formulas and Cockcroft-Gault CrCl renal adjustments.
3. **Visualize Population Genetics**: Analyze Hardy-Weinberg Equilibrium (HWE) and phenotype-specific plasma drug concentrations inline.

## 1. Environment & Database Setup
Importing our core clinical engines and connecting to the SQLite clinical database.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
for sub in ["", "PHAI-101_Python", "PHAI-103_Biostatistics", "PHAI-104_Genomics", "capstone"]:
    p = str(PROJECT_ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom engines
from drug_dosage_calculator import Patient, Drug, calculate_dose, cockcroft_gault
from capstone.interaction_engine import check_drug_interaction, get_connection
from hardy_weinberg import GenotypeData, hardy_weinberg_test, batch_hwe_analysis
from concentration_regression import generate_demo_dataset, anova_by_phenotype

print("✅ Clinical intelligence engines successfully loaded!")

---
## 2. Clinical Case 1: Complex DDI + Pharmacogenomic Risk

### 📋 Patient Profile:
* **Age / Sex**: 42-year-old female
* **Current Medication**: Fluoxetine (Prozac) 20 mg PO daily for Major Depressive Disorder
* **New Prescription**: Codeine 30 mg PO q4-6h PRN for acute post-dental pain
* **Pharmacogenomic Status**: Known CYP2D6 Poor Metabolizer (PM) (*4/*4 genotype)

In [ ]:
# Execute multi-layered interaction analysis
clinical_review = check_drug_interaction(
    drug_a="Codeine",
    drug_b="Fluoxetine",
    patient_genotype="PM",
    gene="CYP2D6"
)

print("=" * 60)
print(f"INTERACTION AUDIT: {clinical_review['drug_a']} + {clinical_review['drug_b']}")
print(f"Risk Tier: {clinical_review['risk']['risk_tier']} (Total Score: {clinical_review['risk']['total_score']})")
print("=" * 60)

for alert in clinical_review['alerts']:
    print(f"  • {alert}")

print("\nClinical Recommendation:")
print(f"  {clinical_review['recommendation_summary']}")

### 💡 Clinical Pharmacist Discussion:
* **Mechanism**: Codeine is a prodrug with negligible analgesic effect; it requires O-demethylation via **CYP2D6** into active **morphine**.
* **Genetic Impact**: As a Poor Metabolizer (PM), the patient cannot convert codeine to morphine $\rightarrow$ **therapeutic failure** and unrelieved pain.
* **Drug Interaction**: Fluoxetine is a potent CYP2D6 inhibitor, which would phenocopy even normal metabolizers into poor metabolizers.
* **Actionable Alternative**: Prescribe a non-opioid (e.g., Ibuprofen 600 mg + Acetaminophen 500 mg) or an opioid not reliant on CYP2D6 (e.g., Morphine, Hydromorphone).

---
## 3. Clinical Case 2: Precision Weight & Renal Dosing Engine

Comparing dosing across two distinct patient populations:
1. **Pediatric Patient**: Standard weight-based calculation.
2. **Geriatric Patient with CKD 4**: CrCl adjustment based on Cockcroft-Gault formula.

In [ ]:
amox = Drug(name="Amoxicillin", standard_dose_mg_per_kg=25.0, max_dose_mg=1000.0, renal_adjustment=True)

# 1. Pediatric: 8 years old, 24 kg, normal kidney function
pediatric_pt = Patient(name="Leo (Pediatric)", age=8, weight_kg=24.0, height_cm=125.0, sex="M", serum_creatinine=0.5)
ped_result = calculate_dose(pediatric_pt, amox)

# 2. Geriatric with CKD 4: 78 years old, 52 kg, SCr 1.8 mg/dL
geriatric_pt = Patient(name="Eleanor (Geriatric CKD 4)", age=78, weight_kg=52.0, height_cm=158.0, sex="F", serum_creatinine=1.8)
ger_result = calculate_dose(geriatric_pt, amox)

dosing_comparison = pd.DataFrame([
    {
        "Patient": ped_result["patient"],
        "Age": ped_result["age_years"],
        "CrCl (mL/min)": ped_result["crcl_mL_min"],
        "Renal Stage": ped_result["renal_stage"],
        "Base Dose (mg)": ped_result["base_dose_mg"],
        "Adjustment Factor": ped_result["renal_adjustment_factor"],
        "Final Dose (mg)": ped_result["final_dose_mg"],
    },
    {
        "Patient": ger_result["patient"],
        "Age": ger_result["age_years"],
        "CrCl (mL/min)": ger_result["crcl_mL_min"],
        "Renal Stage": ger_result["renal_stage"],
        "Base Dose (mg)": ger_result["base_dose_mg"],
        "Adjustment Factor": ger_result["renal_adjustment_factor"],
        "Final Dose (mg)": ger_result["final_dose_mg"],
    }
])

dosing_comparison

---
## 4. Biostatistics: CYP2D6 Population Genetics (HWE Analysis)

Testing whether observed genotype frequencies in major continental populations follow **Hardy-Weinberg Equilibrium** ($p^2 + 2pq + q^2 = 1$).

In [ ]:
demo_populations = [
    GenotypeData("CYP2D6", "*1 (Ref)", "European",       n_AA=740, n_Aa=220, n_aa=40),
    GenotypeData("CYP2D6", "*1 (Ref)", "African",        n_AA=600, n_Aa=310, n_aa=90),
    GenotypeData("CYP2D6", "*1 (Ref)", "East Asian",     n_AA=820, n_Aa=160, n_aa=20),
    GenotypeData("CYP2D6", "*1 (Ref)", "South Asian",    n_AA=710, n_Aa=240, n_aa=50),
]

hwe_df = batch_hwe_analysis(demo_populations)
display_cols = ["population", "N", "p_allele_freq", "q_allele_freq", "chi2_statistic", "p_value", "in_HWE"]
hwe_df[display_cols]

---
## 5. Pharmacokinetic Distribution by Metabolizer Phenotype

Visualizing simulated morphine trough concentrations (ng/mL) across CYP2D6 phenotypes:
* **PM (Poor)**: Substrate accumulation or lack of active metabolite
* **IM (Intermediate)**: Reduced clearance/activity
* **NM (Normal)**: Baseline therapeutic window
* **UM (Ultrarapid)**: Rapid bioactivation $\rightarrow$ risk of opioid toxicity

In [ ]:
pk_data = generate_demo_dataset()
anova_result = anova_by_phenotype(pk_data)

print(anova_result["interpretation"])

# Inline Boxplot
plt.figure(figsize=(9, 5))
colors = {"PM": "#E63946", "IM": "#F4A261", "NM": "#2A9D8F", "UM": "#457B9D"}

sns.boxplot(
    data=pk_data, x="phenotype", y="concentration",
    hue="phenotype", legend=False,
    order=["PM", "IM", "NM", "UM"],
    palette=colors, width=0.5
)
sns.stripplot(
    data=pk_data, x="phenotype", y="concentration",
    hue="phenotype", legend=False,
    order=["PM", "IM", "NM", "UM"],
    palette=colors, size=5, alpha=0.6, jitter=True
)

plt.title("Plasma Morphine Trough Concentrations by CYP2D6 Phenotype", fontsize=13, fontweight="bold")
plt.xlabel("CYP2D6 Metabolizer Status", fontsize=11)
plt.ylabel("Trough Concentration (ng/mL)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

---
## 6. Summary & Next Steps for Portfolio

✅ **Delivered in this notebook**:
- Direct query integration with `pharmacy_intelligence.db`.
- High-risk clinical alert generation combining DDI mechanisms with CPIC Level A recommendations.
- Automated pediatric and CrCl-adjusted renal dosing calculations.
- Inline statistical testing and publication-ready PK visualizations.

**Next Phase**: Connect the OpenFDA API ingestion pipeline to pull real-time MedWatch adverse event reports directly into this workflow.